Tools

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [2]:
from requests.models import Response
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:llama-3.3-70b-versatile")
response=model.invoke("Why do parrots talks")
response

AIMessage(content="Parrots are renowned for their ability to mimic human speech and other sounds, but why do they do it? The reasons behind parrot vocalizations are complex and multifaceted. Here are some possible explanations:\n\n1. **Communication**: In the wild, parrots use vocalizations to communicate with other parrots, warning them of potential predators, signaling food sources, or expressing emotions like excitement or alarm. They may also use vocalizations to establish dominance or attract a mate.\n2. **Social bonding**: Parrots are highly social birds that thrive on interaction. They may learn to mimic human speech as a way to bond with their owners, seeking attention, affection, or rewards. By mimicking human voices, parrots can create a sense of connection and belonging.\n3. **Learning and exploration**: Parrots are intelligent birds with a strong desire to learn and explore their environment. They may experiment with different sounds, including human speech, to understand t

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a location"""
    return f"it's suuny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [5]:
response=model_with_tools.invoke("what's the weather like in boston")
print(response)

for tool_call in response.tool_calls:
    print(f"tool:{tool_call['name']}")
    print(f"args:{tool_call['args']}")



content='' additional_kwargs={'tool_calls': [{'id': 'ddp2d2cfa', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 220, 'total_tokens': 234, 'completion_time': 0.03806446, 'completion_tokens_details': None, 'prompt_time': 0.011912938, 'prompt_tokens_details': None, 'queue_time': 0.056955971, 'total_time': 0.049977398}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e595a-fd3c-7741-8fef-9c27668b5924-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'ddp2d2cfa', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 14, 'total_tokens': 234}
tool:get_weather
args:{'location': 'Boston'}


Tool Execution Loops


In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

I'm glad you're interested in the weather. However, I should let you know that my previous response was just a simulation, and I don't have have access to real-time information. If you want to get the current weather in Boston, I can suggest some ways for you to find out. 

You can check online weather websites or apps, such as AccuWeather or Weather.com, which provide up-to-date weather forecasts and conditions for locations all around the world. You can also check the website of a local news station or newspaper in Boston for the latest weather forecast. 

If you want to get the weather using the functions I have access to, you can use the get_weather function. 




In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '09m17be1x', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.053479346, 'completion_tokens_details': None, 'prompt_time': 0.010676145, 'prompt_tokens_details': None, 'queue_time': 0.056376964, 'total_time': 0.064155491}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e595d-51b5-71e3-9980-55785e0ef27b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '09m17be1x', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}),
 ToolMessage(content="it's